# PhishGuard-PK v2 | One-click Google Colab launcher
This launches English, Roman Urdu and Urdu screening, including batch CSV. Upload the **v2 Full Software ZIP** when prompted. Do not paste sensitive or personal information into public links.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, shutil, sys
print("Upload PHISHGUARD_V2_FULL_SOFTWARE.zip")
files_uploaded=files.upload()
archives=[n for n in files_uploaded if n.lower().endswith(".zip")]
assert archives, "Upload the v2 full software ZIP file"
root=Path("/content/PhishGuard_PK_v2_Full_Software")
root.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    if any(".." in Path(n).parts or n.startswith("/") for n in archive.namelist()):
        raise ValueError("Unsafe ZIP path found")
    archive.extractall(root)
print("Project extracted:",root)

In [ ]:
%pip -q install -r /content/PhishGuard_PK_v2_Full_Software/requirements.txt

In [ ]:
import os,sys
from pathlib import Path
root=Path("/content/PhishGuard_PK_v2_Full_Software")
os.chdir(root)
sys.path.insert(0,str(root))
from fastapi.testclient import TestClient
from phishguard.api import app
c=TestClient(app)
print("Website/API status:",c.get("/health").json())
print("Model status:",c.get("/model-status").json())

In [ ]:
import subprocess,sys,time,requests,os
from google.colab import output
log=open('/content/phishguard_v2_web.log','w')
server=subprocess.Popen([sys.executable,'run_web.py'],cwd=str(root),stdout=log,stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get('http://127.0.0.1:8000/health',timeout=1).status_code==200:break
    except Exception:pass
    time.sleep(1)
else:
    raise RuntimeError('Local website did not start, inspect /content/phishguard_v2_web.log')
try:
    full_link=output.eval_js('google.colab.kernel.proxyPort(8000)')
    print('FULL WEBSITE LINK:',full_link)
    print('This private link works while your Colab runtime stays active.')
except Exception as e:
    print('Colab proxy not available:',type(e).__name__,str(e)[:150])
    print('Use the public Gradio backup link from the next cell.')

In [ ]:
from phishguard.ui import build_ui
print("Use FULL WEBSITE LINK above for the complete responsive app. The PUBLIC Gradio link below is a backup. Keep this cell running while you use the app.")
print("Never paste real OTPs/passwords or private messages into a public link.")
build_ui().launch(share=True,debug=True)